# Flag Smoke Test
Minimal run (2 epochs, 512 train samples) to verify augmentation feeder, weighted sampler, --bbox_norm, and --warmup_epochs flags work end-to-end.\n

**Docker**: run `bash setup.sh` first, then copy `.env.example` → `.env` and set your paths.  
**Colab**: no `.env` needed — defaults kick in automatically.

In [ ]:
# ── Cell 1: Load environment ──────────────────────────────────────────────────
# On Docker: reads /workspace/Yolo-ST-GCN/.env (set by Duy after cp .env.example .env)
# On Colab:  .env absent → load_dotenv() is a no-op → os.getenv falls back to /content/... defaults

import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv

# Look for .env next to the notebook, then at repo root one level up
_nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
for _candidate in [_nb_dir / '.env', _nb_dir.parent / '.env', Path('.env')]:
    if _candidate.exists():
        load_dotenv(_candidate)
        print(f'Loaded .env from {_candidate}')
        break
else:
    print('No .env found — using built-in Colab defaults')

REPO_DIR   = os.getenv('REPO_DIR',   '/content/Yolo-ST-GCN')
BRANCH     = os.getenv('BRANCH',     'refactor-1')
GYM288_PKL = os.getenv('GYM288_PKL', '/content/Gym288-skeleton/gym_288_skeleton.pkl')
GYM99_PKL  = os.getenv('GYM99_PKL',  '/content/Gym99-from-Gym288/gym99_from_gym288.pkl')
OUT_DIR    = os.getenv('OUT_DIR',    'outputs/flag_smoke_test')

print(f'REPO_DIR   = {REPO_DIR}')
print(f'BRANCH     = {BRANCH}')
print(f'GYM288_PKL = {GYM288_PKL}')
print(f'GYM99_PKL  = {GYM99_PKL}')
print(f'OUT_DIR    = {OUT_DIR}')

In [ ]:
# ── Cell 2: Repo setup ────────────────────────────────────────────────────────
# Docker: repo already cloned by setup.sh → just pull latest
# Colab:  repo not present → clone it

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'

if not Path(REPO_DIR).exists():
    print('Cloning repo...')
    subprocess.run(
        ['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 3: Dataset ───────────────────────────────────────────────────────────
# Docker: GYM288_PKL already on disk → skip download
# Colab:  download from HuggingFace

if Path(GYM288_PKL).exists():
    print(f'Gym288 pickle found: {GYM288_PKL}')
else:
    print('Gym288 pickle not found — downloading from HuggingFace...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'], check=True)
    from huggingface_hub import snapshot_download
    download_dir = Path(GYM288_PKL).parent
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
    pkl_candidates = sorted(download_dir.rglob('*.pkl'))
    if not pkl_candidates:
        raise FileNotFoundError('No .pkl found after Gym288 download.')
    GYM288_PKL = str(pkl_candidates[0])
    print(f'Downloaded: {GYM288_PKL}')

In [ ]:
# ── Cell 4: Smoke train ───────────────────────────────────────────────────────
# 2 epochs so warmup_epochs=1 actually fires the LinearLR→CosineAnnealingLR transition.
# 512 train / 128 val samples — just enough to touch all code paths.

cmd = [
    'python', 'scripts/train_gym99.py',
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--out_dir',             OUT_DIR,
    '--epochs',              '2',
    '--batch_size',          '64',
    '--lr',                  '0.001',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--use_two_stream',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--max_train_samples',   '512',
    '--max_val_samples',     '128',
    '--bbox_norm',
    '--warmup_epochs',       '1',
    '--use_augment_feeder',
    '--use_weighted_sampler',
]

print('Command:')
print(' '.join(cmd))
print()

result = subprocess.run(cmd, check=True)
print('\nSmoke run finished with exit code', result.returncode)

In [ ]:
# ── Cell 5: Verify output ─────────────────────────────────────────────────────
import json

metrics_path = Path(OUT_DIR) / 'metrics_train_gym99.json'
history_path = Path(OUT_DIR) / 'history.json'

print('=== metrics ===')
with open(metrics_path) as f:
    print(json.dumps(json.load(f), indent=2))

print('\n=== history (last epoch) ===')
with open(history_path) as f:
    h = json.load(f)
for k, v in h.items():
    print(f'  {k}: {v}')

print('\nAll output files:')
for p in sorted(Path(OUT_DIR).iterdir()):
    print(' ', p.name)

In [ ]:
# ── Cell 6: Confusion Matrix Inference ────────────────────────────
# Evaluate the trained weights on both train and val subsets (unaugmented)
# to generate the Confusion Matrices.

import os
import torch
from torch.utils.data import DataLoader
from IPython.display import Image, display

from src.gym99_dataset import build_gym99_data_tensors, infer_num_gym99_classes
from src.dataset import PennActionDataset
from src.two_stream_stgcn import TwoStream_STGCN
from src.model import Model_STGCN
from src.losses import build_classification_criterion
from src.train import eval_epoch
from src.visualize import plot_confusion_matrix
from src.skeleton_utils import bbox_normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_two_stream = '--use_two_stream' in cmd
joint_spec = 'coco18'

print('1/4. Loading dataset tensors...')
data, bone_data, labels, flags, _, _ = build_gym99_data_tensors(
    dataset_path=GYM99_PKL,
    joint_spec_name=joint_spec,
    split='all',
    keep_unknown_split=False,
    return_bone_data=use_two_stream
)

train_mask = flags == 1
test_mask = flags == 0
X_train, y_train = bbox_normalize(data[train_mask]), labels[train_mask]
X_val, y_val = bbox_normalize(data[test_mask]), labels[test_mask]
B_train = B_val = None
if use_two_stream:
    B_train, B_val = bbox_normalize(bone_data[train_mask]), bbox_normalize(bone_data[test_mask])

# Limit evaluation to the exact same subset used during training so inference doesn't take forever
N_TRAIN, N_VAL = 512, 128
X_train, y_train = X_train[:N_TRAIN], y_train[:N_TRAIN]
X_val,   y_val   = X_val[:N_VAL],   y_val[:N_VAL]
if use_two_stream:
    B_train, B_val = B_train[:N_TRAIN], B_val[:N_VAL]

train_ds = PennActionDataset(X_train, y_train, bone_data=B_train, include_bone=use_two_stream, joint_spec_name=joint_spec)
val_ds   = PennActionDataset(X_val, y_val, bone_data=B_val, include_bone=use_two_stream, joint_spec_name=joint_spec)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)

print('2/4. Loading model & weights...')
num_classes = infer_num_gym99_classes(GYM99_PKL)
model = (
    TwoStream_STGCN(num_classes=num_classes, joint_spec=joint_spec)
    if use_two_stream
    else Model_STGCN(num_classes=num_classes, joint_spec=joint_spec)
).to(device)

weights_name = 'stgcn_gym99_coco18_2s.pth' if use_two_stream else 'stgcn_gym99_coco18.pth'
weights_path = os.path.join(OUT_DIR, weights_name)
model.load_state_dict(torch.load(weights_path, map_location=device)['model_state_dict'])

eval_criterion = build_classification_criterion('ce', device)
classes_list = [str(i) for i in range(num_classes)]

print('3/4. Evaluating Train Set...')
_, _, _, train_preds, train_gt = eval_epoch(model, train_loader, eval_criterion, device)
plot_confusion_matrix(train_gt, train_preds, classes=classes_list, title='Confusion Matrix — Train Set', out_dir=OUT_DIR, filename='train_cm.png')

print('4/4. Evaluating Val Set...')
_, _, _, val_preds, val_gt = eval_epoch(model, val_loader, eval_criterion, device)
plot_confusion_matrix(val_gt, val_preds, classes=classes_list, title='Confusion Matrix — Val Set', out_dir=OUT_DIR, filename='val_cm.png')

print('\n\u2500\u2500\u2500\u2500 TRAIN CONFUSION MATRIX \u2500\u2500\u2500\u2500')
display(Image(filename=os.path.join(OUT_DIR, 'train_cm.png')))
print('\n\u2500\u2500\u2500\u2500 VAL CONFUSION MATRIX \u2500\u2500\u2500\u2500')
display(Image(filename=os.path.join(OUT_DIR, 'val_cm.png')))
